# Colab Frontend - Validation / Prediction Accuracy (Qwen3-1.7B, TPU)

Clones the repo, sets parameters, and runs `validation/validation_unified.ipynb` via papermill, then shows accuracy/F1/confusion-matrix figures. **Select a TPU runtime.** You must have pushed `validation/validation_unified.ipynb` to the `colab` branch.

In [ ]:
# Install papermill + helpers. (Select a TPU runtime: Runtime > Change runtime type > TPU.)
!pip install -q papermill nest_asyncio nbformat nbconvert ipykernel
print("papermill + helpers installed.")


In [ ]:
import os, subprocess, shutil

REPO_URL = "https://github.com/HangYu8123/SC_Ageing_Prediction.git"
REPO_BRANCH = "colab"          # branch that contains the *_unified.ipynb notebooks
PROJECT_DIR = "/content/SC_Ageing_Prediction"
UNIFIED_NB = os.path.join(PROJECT_DIR, "validation/validation_unified.ipynb")

# Mount Drive in THIS (interactive) kernel; the papermill child shares the FUSE mount.
try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
        print("Drive mounted.")
except Exception as exc:
    print("Drive not mounted (non-Colab?):", repr(exc))

# Read the HF token from Colab Secrets HERE and pass it via the environment (never hardcode).
# (userdata only works in this interactive kernel, not inside the papermill child.)
try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        print("HF_TOKEN loaded from Colab Secrets into the environment.")
    else:
        print("No HF_TOKEN secret set (only needed for pushing to the Hub).")
except Exception as exc:
    print("HF_TOKEN secret not available:", repr(exc))

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, PROJECT_DIR])
assert os.path.exists(UNIFIED_NB), (
    f"Unified notebook not found: {UNIFIED_NB}.\n"
    f"Commit and push the *_unified.ipynb files to the '{REPO_BRANCH}' branch first."
)
print("Cloned", REPO_URL, "@", REPO_BRANCH)
print("Target notebook:", UNIFIED_NB)


## Parameters
Edit `PARAMS` to choose the variant. Commented lines show the other presets.

In [ ]:
PARAMS = {
    # ---- 1K variant (default) ----
    "MODEL_ID": "DaisyCuttie/QWEN3-1.7B-EIGHT-ORGANS-EXTENDED-1K",
    "MODEL_LABEL": "1K Length Model",
    "TRUNCATE_GENES": True,
    "PLOT_R2_SUMMARY": False,
    "COPY_TO_DRIVE": False,
    "OUTPUT_DIR": "/content/qwen3_prediction_acc_outputs",
    "PROJECT_DIR": PROJECT_DIR,
    # ---- Full-length variant: uncomment to override ----
    # "MODEL_ID": "DaisyCuttie/QWEN3-1.7B-EIGHT-ORGANS-EXTENDED-FULL-LENGTH",
    # "MODEL_LABEL": "Full Length Model",
    # "TRUNCATE_GENES": False,
    # ---- Full + organ R-squared summary: also set ----
    # "PLOT_R2_SUMMARY": True,
}
print(PARAMS)


## Run the notebook via papermill

In [ ]:
import os, logging, nest_asyncio, papermill as pm

nest_asyncio.apply()                 # Colab already runs an event loop; papermill needs this.
logging.basicConfig(level=logging.INFO)   # stream papermill/cell logs live below.

OUTPUT_NB = "/content/validation_unified.executed.ipynb"

pm.execute_notebook(
    UNIFIED_NB,
    OUTPUT_NB,
    parameters=PARAMS,
    kernel_name="python3",
    log_output=True,
    progress_bar=False,
)
print("\nExecution finished. Output notebook:", OUTPUT_NB)


## Show the results

In [ ]:
# Show the results produced by the executed notebook.
import os, glob, json
from IPython.display import Image, HTML, display

# 1) Any figures saved by the run.
shown = 0
for d in [PARAMS.get("OUTPUT_DIR")]:
    if d and os.path.isdir(d):
        for png in sorted(glob.glob(os.path.join(d, "*.png"))):
            print(png)
            display(Image(filename=png)); shown += 1
print(f"Displayed {shown} saved figure(s).")

# 2) Any results JSON.
for d in [PARAMS.get("OUTPUT_DIR")]:
    if d and os.path.isdir(d):
        for js in sorted(glob.glob(os.path.join(d, "*.json"))):
            print("\n==", js, "==")
            with open(js) as f:
                data = json.load(f)
            print(json.dumps(data, indent=2)[:2000])

# 3) Full executed notebook rendered inline (figures included; may be large).
import nbformat
from nbconvert import HTMLExporter
nb = nbformat.read(OUTPUT_NB, as_version=4)
exporter = HTMLExporter(); exporter.exclude_input = True
body, _ = exporter.from_notebook_node(nb)
display(HTML(body))
